In [1]:
import os
import json
import random
import numpy as np
import torch

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
!git clone https://github.com/lxcentry/cu_konk_stip_ml_red_task.git

Cloning into 'cu_konk_stip_ml_red_task'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 37 (delta 4), reused 29 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 5.24 MiB | 12.12 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [3]:
!pip install -q -U transformers accelerate peft trl bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 8.4 MB/s eta 0:00:00


In [4]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen3-4B-Instruct-2507",
    local_dir="./qwen3_4b",
    local_dir_use_symlinks=False,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

'/content/qwen3_4b'

In [5]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

MODEL_NAME = "./qwen3_4b"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [6]:
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 132,120,576 || all params: 4,154,588,672 || trainable%: 3.1801


In [7]:
import json
from datasets import Dataset

examples = []

with open("cu_konk_stip_ml_red_task/ml-olympiad-red-task/data/kid_adult.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        messages = [
            {"role": "user", "content": row["prompt"]},
            {"role": "assistant", "content": row["kid"]},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        examples.append({"text": text})

train_dataset = Dataset.from_list(examples)

print(train_dataset[0])

{'text': '<|im_start|>user\nКак работает камера видеонаблюдения? Ответ: Она ловит свет через стеклышко, превращает его в электрические сигналы и сохраняет их как видео в памяти.<|im_end|>\n<|im_start|>assistant\nКамера работает как волшебный глаз. Она улавливает свет через свое стеклышко, переводит его в электрические сигналы и сохраняет как видео в памяти.<|im_end|>\n'}


In [8]:
from trl import SFTTrainer
import inspect

print(inspect.signature(SFTTrainer))

(model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[transformers.trainer_callback.TrainerCallback] | None =

In [10]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./sft_model",

    learning_rate=2e-4,

    num_train_epochs=1,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=8,

    logging_steps=10,

    save_strategy="epoch",

    max_length=512,

    seed=42,

    bf16=True,
    fp16=False,
)

In [11]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [12]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


KeyboardInterrupt: 

In [ ]:
trainer.save_model("./sft_adapter")

tokenizer.save_pretrained("./sft_adapter")

In [ ]:
import json

test = []

with open("/content/cu_konk_stip_ml_red_task/ml-olympiad-red-task/data/public_test_style.jsonl", encoding="utf-8") as f:
    for line in f:
        test.append(json.loads(line))

print(test[0])

In [ ]:
import joblib

style_clf = joblib.load("/content/cu_konk_stip_ml_red_task/ml-olympiad-red-task/metrics/style_clf.pkl")

print(type(style_clf))
print(style_clf)

In [ ]:
import joblib
from scipy.sparse import hstack

style = joblib.load("/content/cu_konk_stip_ml_red_task/ml-olympiad-red-task/metrics/style_clf.pkl")

clf = style["clf"]
word_vec, char_vec = style["vecs"]

In [ ]:
from tqdm import tqdm

predictions = []

model.eval()

for sample in tqdm(test):

    messages = [
        {"role": "user", "content": sample["prompt"]}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=None,
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    predictions.append(
        {
            "prompt": sample["prompt"],
            "prediction": answer
        }
    )

In [ ]:
with open("predictions_sft.jsonl", "w", encoding="utf-8") as f:
    for row in predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
texts = [x["prediction"] for x in predictions]

In [ ]:
X_word = word_vec.transform(texts)
X_char = char_vec.transform(texts)

X = hstack([X_word, X_char])

In [ ]:
labels = clf.predict(X)

print(labels[:20])

In [ ]:
print(clf.classes_)
print(clf.predict(X[:10]))

In [ ]:
probs = clf.predict_proba(X)

P_simple = probs[:, 1].mean()

print(f"P_simple = {P_simple:.4f}")

Задание 2

In [ ]:
import json
from datasets import Dataset

pairs = []

with open("/content/cu_konk_stip_ml_red_task/ml-olympiad-red-task/data/kid_adult.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        pairs.append(
            {
                "prompt": row["prompt"],
                "chosen": row["kid"],
                "rejected": row["adult"],
            }
        )

dpo_dataset = Dataset.from_list(pairs)

print(dpo_dataset[0])

In [ ]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)

from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "./sft_adapter",
)

In [ ]:
base_ref = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)

ref_model = PeftModel.from_pretrained(
    base_ref,
    "./sft_adapter",
)

ref_model.eval()

In [ ]:
from trl import DPOConfig

dpo_args = DPOConfig(

    output_dir="./dpo_model",

    learning_rate=5e-6,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=8,

    num_train_epochs=1,

    logging_steps=10,

    save_strategy="epoch",

    max_length=512,

    beta=0.1,

    bf16=True,
    fp16=False,

    report_to="none",

    seed=42,
)

In [ ]:
from trl import DPOTrainer

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./dpo_adapter")

tokenizer.save_pretrained("./dpo_adapter")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(
    base_model,
    "./dpo_adapter"
)

model.eval()

In [ ]:
predictions = []

for sample in test:

    messages = [
        {
            "role":"user",
            "content":sample["prompt"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    predictions.append(
        {
            "prompt":sample["prompt"],
            "prediction":answer,
        }
    )

In [ ]:
import joblib
import numpy as np
from scipy.sparse import hstack

style = joblib.load("metrics/style_clf.pkl")

clf = style["clf"]
word_vec, char_vec = style["vecs"]

texts = [x["prediction"] for x in predictions]

X_word = word_vec.transform(texts)
X_char = char_vec.transform(texts)

X = hstack([X_word, X_char])

probs = clf.predict_proba(X)

P_simple = probs[:, 1].mean()

print(f"P_simple = {P_simple:.4f}")